In [1]:
!pip install tensorflow==2.12.0

In [2]:
import tensorflow as tf

print(tf.__version__)

2.12.0


In [3]:
!pip install keras

In [4]:
!pip install keras-rl2

In [5]:
!pip install gym

In [6]:
!pip install numpy

In [7]:
import gym
from gym import Wrapper
import numpy as np

In [8]:
from tensorflow.keras.models import Sequential
from keras.layers import Dense, Flatten
from keras.optimizers import Adam

In [9]:
from rl.agents.dqn import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory

In [12]:
# ساخت Wrapper برای هماهنگ کردن خروجی با نسخه جدید gym
class MyGymWrapper(Wrapper):
    def reset(self, **kwargs):
        state, info = self.env.reset(**kwargs)
        return state  # فقط حالت برگردان

    def step(self, action):
        state, reward, done, truncated, info = self.env.step(action)
        done = done or truncated  # ترکیب done و truncated
        return state, reward, done, info

    def render(self, *args, **kwargs):
        # حذف آرگومان mode که در نسخه‌های جدید gym قبول نمی‌شود
        return self.env.render()

In [13]:
# ساخت محیط و بسته‌بندی با Wrapper
# env = gym.make('CartPole-v1')
env = gym.make('CartPole-v1', render_mode='human')
env = MyGymWrapper(env)

In [14]:
# ابعاد ورودی (مشاهده) و تعداد اکشن‌ها
nb_actions = env.action_space.n
input_shape = env.observation_space.shape  # معمولاً (4,)

print("Observation space shape:", input_shape)
print("Number of actions:", nb_actions)

Observation space shape: (4,)
Number of actions: 2


In [15]:
# ساخت مدل شبکه عصبی
model = Sequential()
model.add(Flatten(input_shape=(1,) + input_shape))  # توجه به شکل (1, 4)
model.add(Dense(48, activation='relu'))
model.add(Dense(24, activation='relu'))
model.add(Dense(nb_actions, activation='linear'))
print(model.summary())

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_2 (Flatten)         (None, 4)                 0         
                                                                 
 dense_6 (Dense)             (None, 48)                240       
                                                                 
 dense_7 (Dense)             (None, 24)                1176      
                                                                 
 dense_8 (Dense)             (None, 2)                 50        
                                                                 
Total params: 1,466
Trainable params: 1,466
Non-trainable params: 0
_________________________________________________________________
None


In [16]:
# حافظه و سیاست
memory = SequentialMemory(limit=50000, window_length=1)
policy = BoltzmannQPolicy()

In [17]:
# ساخت عامل DQN
dqn = DQNAgent(model=model, nb_actions=nb_actions, memory=memory,
               nb_steps_warmup=10, target_model_update=1e-2,
               policy=policy)

In [18]:
# کامپایل کردن مدل
dqn.compile(Adam(learning_rate=1e-3), metrics=['mae'])

In [19]:
# آموزش مدل
dqn.fit(env, nb_steps=5000, visualize=False, verbose=1)

Training for 5000 steps ...


ValueError: too many values to unpack (expected 2)

In [20]:
# تست مدل
dqn.test(env, nb_episodes=5, visualize=True)

Testing for 5 episodes ...


ValueError: too many values to unpack (expected 2)